# Task result loader

This notebook opens an existing task execution result store with `tasks.dispatch.load_result_zarr(...)` and inspects its contents. By default it selects the most recent `.zarr` store under `artifacts/notebook_runs/`; override `result_store_path` below if you want a specific run.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd()
if not (project_root / "network_creation").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    import scienceplots  # noqa: F401
except ModuleNotFoundError:
    scienceplots = None
else:
    plt.style.use("science")

from tasks.dispatch import load_result_zarr

project_root


PosixPath('/home/hdahmou/Network_creation')

## Select a result store

The cell below picks the newest `.zarr` store under `artifacts/notebook_runs/`. Replace `result_store_path` afterward if you want to inspect a different run.

In [3]:
notebook_run_dir = project_root / "artifacts" / "notebook_runs"
result_store_candidates = sorted(
    notebook_run_dir.glob("*/*.zarr"),
    key=lambda path: path.stat().st_mtime,
)
if not result_store_candidates:
    raise FileNotFoundError(f"No result stores found under {notebook_run_dir}")

result_store_path = result_store_candidates[-1]

{
    "result_store_path": str(result_store_path.resolve()),
    "candidate_count": len(result_store_candidates),
}


FileNotFoundError: No result stores found under /home/hdahmou/Network_creation/artifacts/notebook_runs

## Load metadata and outputs

In [ ]:
result_data = load_result_zarr(result_store_path)
input_index = result_data["input_index"]
stored_outputs = result_data["outputs"]
failure_arrays = result_data["failures"]

{
    "result_store_path": str(result_data["path"]),
    "schema_version": result_data["schema_version"],
    "task_name": result_data["task_name"],
    "task_class": result_data["task_class"],
    "output_schema": result_data["output_schema"],
    "total_inputs": result_data["total_inputs"],
    "stored_results": result_data["stored_results"],
    "stored_failures": result_data["stored_failures"],
}


In [ ]:
{name: values.shape for name, values in stored_outputs.items()}


## Plot per-step outputs

Only outputs with shape `(stored_results, n_steps)` are plotted automatically.

In [ ]:
plot_ready_outputs = {
    name: values
    for name, values in stored_outputs.items()
    if values.ndim == 2
}

if not plot_ready_outputs:
    raise ValueError("No per-step outputs with shape (stored_results, n_steps) were found.")

n_plots = len(plot_ready_outputs)
fig, axes = plt.subplots(n_plots, 1, figsize=(12, 4 * n_plots), sharex=False)
axes = np.atleast_1d(axes)

ordered_input_index = input_index
for axis, (name, values) in zip(axes, plot_ready_outputs.items(), strict=True):
    steps = np.arange(values.shape[1])
    for row_index, sample_idx in enumerate(ordered_input_index):
        axis.plot(steps, values[row_index], alpha=0.8, label=f"input {int(sample_idx)}")
    axis.set_title(name)
    axis.set_xlabel("step")
    axis.set_ylabel("value")
    axis.grid(alpha=0.3)
    if len(ordered_input_index) <= 10:
        axis.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))

plt.tight_layout()
plt.show()


## Final-step summary

In [ ]:
summary = {"input_index": ordered_input_index}
for name, values in stored_outputs.items():
    if values.ndim >= 2:
        summary[f"{name}.final"] = values[:, -1]
    else:
        summary[name] = values

summary


## Failure rows

If the task recorded failures, this cell shows the raw arrays stored under the `failures/` group.

In [ ]:
failure_arrays if failure_arrays else "No stored failures."
